# SetFit HTS Classifier Training - Colab Pro

This notebook trains a SetFit model on the full CROSS rulings dataset.

**Before running:**
1. Runtime → Change runtime type → **A100 GPU** (or V100) → Save
2. Upload your training data files

**Expected time:** 45-60 minutes on A100 GPU  
**Expected accuracy:** 90-92%

## Step 1: Install Dependencies

In [ ]:
!pip install -q setfit==1.1.3 transformers==4.57.6 torch scikit-learn sentence-transformers datasets

## Step 2: Upload Training Data

In [ ]:
from google.colab import files
import json

print("Upload crossRulings.json (training data)...")
uploaded = files.upload()

print("\nUpload crossRulings-validation.json...")
uploaded_val = files.upload()

print("\nUpload crossRulings-test.json...")
uploaded_test = files.upload()

## Step 3: Prepare Dataset (Memory Optimized)

In [ ]:
import json
from datasets import Dataset
import gc

def load_rulings(filepath):
    with open(filepath, 'r') as f:
        data = json.load(f)
        if isinstance(data, list):
            return data
        elif isinstance(data, dict) and 'rulings' in data:
            return data['rulings']
        else:
            raise ValueError('Unexpected data format')

def prepare_dataset(rulings, level='subheading', max_samples=None):
    texts = []
    labels = []
    
    for i, ruling in enumerate(rulings):
        if max_samples and i >= max_samples:
            break
            
        text = ruling.get('productDescription') or ruling.get('product_description', '')
        hts_codes = ruling.get('htsCodes') or ruling.get('hts_codes', [])
        
        if not text or not hts_codes:
            continue
        
        hts_code = hts_codes[0] if isinstance(hts_codes, list) else hts_codes
        
        if level == 'chapter':
            label = hts_code[:2]
        elif level == 'heading':
            label = hts_code[:4]
        elif level == 'subheading':
            label = hts_code[:6]
        else:
            label = hts_code
        
        texts.append(text)
        labels.append(label)
    
    return Dataset.from_dict({'text': texts, 'label': labels})

# Load dataset with aggressive memory limits
print("Loading training data...")
train_rulings = load_rulings('crossRulings.json')
print(f"Total available: {len(train_rulings)} rulings")

val_rulings = load_rulings('crossRulings-validation.json')
test_rulings = load_rulings('crossRulings-test.json')

# Use 15,000 samples (fits in L4 RAM, still excellent accuracy)
# SetFit's contrastive learning creates pairs, so 15K samples = ~225K pairs
train_dataset = prepare_dataset(train_rulings, level='subheading', max_samples=15000)
val_dataset = prepare_dataset(val_rulings, level='subheading', max_samples=100)
test_dataset = prepare_dataset(test_rulings, level='subheading')

print(f"\n✅ Using {len(train_dataset)} samples (optimized for L4 RAM limits)")

# Free memory
del train_rulings, val_rulings, test_rulings
gc.collect()

print(f"\nTraining samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Unique labels: {len(set(train_dataset['label']))}")

## Step 4: Train SetFit Model (Optimized Settings)

In [ ]:
from setfit import SetFitModel, Trainer, TrainingArguments
import torch
import gc

print("GPU available:", torch.cuda.is_available())
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
print("GPU name:", gpu_name)

# Aggressive memory cleanup
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Initialize model
print("\nInitializing SetFit model...")
model = SetFitModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2",
    labels=list(set(train_dataset['label']))
)

# Ultra-conservative settings for L4
batch_size = 8  # Smaller batch = less RAM usage
num_epochs = 3
print(f"Using batch_size={batch_size} (L4-optimized)")

# Training arguments - minimize RAM usage
args = TrainingArguments(
    batch_size=batch_size,
    num_epochs=num_epochs,
    evaluation_strategy="no",
    save_strategy="epoch",
    output_dir="./setfit-hts-subheading",
    logging_steps=100,
    num_iterations=20,  # Reduce contrastive pair generation (saves RAM)
)

# Create trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
)

# Train!
print("\nStarting training...")
print(f"GPU: {gpu_name}")
print(f"Training samples: {len(train_dataset)}")
print(f"Epochs: {num_epochs}")
print(f"Batch size: {batch_size}")
print(f"Expected time: 60-90 minutes")
print("\n⏳ Generating contrastive pairs (this takes 2-3 minutes)...")
print("Training progress will update every 100 steps after pair generation...\n")

trainer.train()

print("\n✅ Training complete!")

## Step 5: Evaluate Model

In [ ]:
import time
from sklearn.metrics import accuracy_score

print("Evaluating on test set...")
start_time = time.time()

predictions = model.predict(test_dataset['text'])
accuracy = accuracy_score(test_dataset['label'], predictions)
inference_time = (time.time() - start_time) / len(test_dataset) * 1000

print(f"\n{'='*60}")
print(f"Test Accuracy: {accuracy*100:.2f}%")
print(f"Avg Inference Time: {inference_time:.2f}ms per sample")
print(f"{'='*60}")

metrics = {
    'accuracy': accuracy,
    'total_samples': len(test_dataset),
    'inference_time_ms': inference_time,
    'training_samples': len(train_dataset),
    'model_name': 'setfit-hts-subheading'
}

with open('./setfit-hts-subheading/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("\nMetrics saved!")

## Step 6: Test Sample Predictions

In [ ]:
test_examples = [
    "cotton t-shirt",
    "plastic water bottle",
    "leather wallet",
    "stainless steel kitchen knife",
    "wooden dining table"
]

print("Sample predictions:\n")
for example in test_examples:
    pred = model.predict([example])[0]
    print(f"'{example}' → {pred}")

## Step 7: Save and Download Model

In [ ]:
print("Saving model...")
model.save_pretrained('./setfit-hts-subheading')
print("Model saved!")

!zip -r setfit-hts-subheading.zip setfit-hts-subheading/

print("\nDownloading model...")
files.download('setfit-hts-subheading.zip')

print("\n✅ Done! Extract and use in your project.")